In [ ]:
import opik
opik.configure(use_local=False)

In [ ]:
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

# Setup Workflow

In [ ]:
import os
from llama_index.llms.ollama import Ollama

def load_llm(model_option):
    if model_option == "Qwen3":
        llm = Ollama(model="qwen3")
    else:
        llm = Ollama(model="deepseek-r1")
    return llm

In [ ]:
model_name = 'Qwen3'
# model_name = 'DeepSeek-R1'
llm  = load_llm(model_name)


# Trace RAG calls 

In [ ]:
from llama_index.core import Settings
from llama_index.core.callbacks import CallbackManager
from opik.integrations.llama_index import LlamaIndexCallbackHandler

# A callback handler tp automatically log all LlamaIndex operations to Opik
opik_callback_handler = LlamaIndexCallbackHandler()

# Integrate handler into LlamaIndex's settings
Settings.callback_manager = CallbackManager([opik_callback_handler])

# Evaluation

In [ ]:
from opik import Opik

client = Opik()
dataset = client.get_or_create_dataset(name="Test dataset")

In [ ]:
import pandas as pd

df = pd.read_csv("./eval-data/test.csv")

In [ ]:
# insert the data into the dataset

qa_pairs = [
    {"input": row["Question"], "expected_output": row["Answer"], "context": row["Context"]} 
    for _, row in df.iterrows()
]
qa_pairs[0]


Use insert if you're creating the dataset for the first time

In [ ]:

# dataset.insert(qa_pairs)

In [ ]:
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader


Settings.llm = llm
Settings.embed_model = FastEmbedEmbedding(model_name="nomic-ai/nomic-embed-text-v1")

documents = SimpleDirectoryReader("./eval-data/paul_graham").load_data()
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine()

In [ ]:
from opik import track

@track
def my_llm_application(input: str) -> str:
    response = query_engine.query(input)
    return str(response)

def evaluation_task(x):
    return {
        "output": my_llm_application(x['input'])
    }

In [ ]:
from opik.evaluation.metrics import (
    Hallucination,
    AnswerRelevance,
    ContextPrecision,
    ContextRecall
)

# Define the metrics
hallucination_metric = Hallucination()
answer_relevance_metric = AnswerRelevance()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall() 

In [ ]:
from opik.evaluation import evaluate

evaluation = evaluate(
    dataset=dataset,
    task=evaluation_task,
    experiment_name = model_name,
    scoring_metrics=[hallucination_metric, answer_relevance_metric, context_precision_metric, context_recall_metric],
    experiment_config={
        "model": "gpt-3.5-turbo"
    }
)